# Button-press-v3 Checkpoint Evaluation Visualization


In [ ]:
from pathlib import Path
import pandas as pd # type: ignore
import numpy as np # type: ignore
import matplotlib.pyplot as plt # type: ignore

# Change this if your notebook is not inside C:/Users/Michalis/Desktop/button_press_v3
ROOT = Path("button-press_v3_ppo_split_runs")
EVAL_DIR = ROOT / "results" / "checkpoint_eval"

summary_path = EVAL_DIR / "button_checkpoint_across_splits_summary.csv"
per_run_path = EVAL_DIR / "button_checkpoint_summary.csv"
raw_path = EVAL_DIR / "button_checkpoint_raw_episodes.csv"

print("Looking for:", summary_path)
assert summary_path.exists(), f"Missing file: {summary_path}"

summary = pd.read_csv(summary_path)
per_run = pd.read_csv(per_run_path) if per_run_path.exists() else None
raw = pd.read_csv(raw_path) if raw_path.exists() else None

summary.head()

## Quick overview

In [ ]:
print("Configs:", sorted(summary["config_name"].unique()))
print("Checkpoints:", sorted(summary["checkpoint_step"].unique()))
print("Groups:", sorted(summary["group"].unique()))
print("Rows:", len(summary))

summary.sort_values(["config_name", "checkpoint_step", "group"]).head(20)

## Success rate over checkpoints

In [ ]:
for group_name in sorted(summary["group"].unique()):
    fig, ax = plt.subplots(figsize=(10, 6))
    data = summary[summary["group"] == group_name]
    
    for config in sorted(data["config_name"].unique()):
        d = data[data["config_name"] == config].sort_values("checkpoint_step")
        ax.plot(
            d["checkpoint_step"],
            d["mean_success_rate"],
            marker="o",
            label=config,
        )
        # Error bars show std across splits
        ax.errorbar(
            d["checkpoint_step"],
            d["mean_success_rate"],
            yerr=d["std_success_rate"].fillna(0),
            fmt="none",
            capsize=3,
        )
    
    ax.set_title(f"Button-press-v3 success rate over checkpoints ({group_name})")
    ax.set_xlabel("Checkpoint timesteps")
    ax.set_ylabel("Mean success rate across splits")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

## First success step over checkpoints

In [ ]:
for group_name in sorted(summary["group"].unique()):
    fig, ax = plt.subplots(figsize=(10, 6))
    data = summary[summary["group"] == group_name]
    
    for config in sorted(data["config_name"].unique()):
        d = data[data["config_name"] == config].sort_values("checkpoint_step")
        ax.plot(
            d["checkpoint_step"],
            d["mean_first_success_step"],
            marker="o",
            label=config,
        )
        ax.errorbar(
            d["checkpoint_step"],
            d["mean_first_success_step"],
            yerr=d["std_first_success_step"].fillna(0),
            fmt="none",
            capsize=3,
        )
    
    ax.set_title(f"Button-press-v3 first success step over checkpoints ({group_name})")
    ax.set_xlabel("Checkpoint timesteps")
    ax.set_ylabel("Mean first success step across splits")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

## Average return over checkpoints


In [ ]:
for group_name in sorted(summary["group"].unique()):
    fig, ax = plt.subplots(figsize=(10, 6))
    data = summary[summary["group"] == group_name]
    
    for config in sorted(data["config_name"].unique()):
        d = data[data["config_name"] == config].sort_values("checkpoint_step")
        ax.plot(
            d["checkpoint_step"],
            d["mean_return"],
            marker="o",
            label=config,
        )
        ax.errorbar(
            d["checkpoint_step"],
            d["mean_return"],
            yerr=d["std_return_across_splits"].fillna(0),
            fmt="none",
            capsize=3,
        )
    
    ax.set_title(f"Button-press-v3 return over checkpoints ({group_name})")
    ax.set_xlabel("Checkpoint timesteps")
    ax.set_ylabel("Mean return across splits")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

## Train vs test comparison at each checkpoint

In [ ]:
for config in sorted(summary["config_name"].unique()):
    fig, ax = plt.subplots(figsize=(10, 6))
    data = summary[summary["config_name"] == config]
    
    for group_name in sorted(data["group"].unique()):
        d = data[data["group"] == group_name].sort_values("checkpoint_step")
        ax.plot(
            d["checkpoint_step"],
            d["mean_success_rate"],
            marker="o",
            label=group_name,
        )
    
    ax.set_title(f"Train vs test success rate: {config}")
    ax.set_xlabel("Checkpoint timesteps")
    ax.set_ylabel("Mean success rate across splits")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

## Compact final tables

In [ ]:
success_table = summary.pivot_table(
    index=["config_name", "checkpoint_step"],
    columns="group",
    values="mean_success_rate",
).reset_index()

first_success_table = summary.pivot_table(
    index=["config_name", "checkpoint_step"],
    columns="group",
    values="mean_first_success_step",
).reset_index()

print("Success rate table")
display(success_table)

print("First success step table")
display(first_success_table)

## Best checkpoint per configuration

For success rate, many configs may tie at 1.0. In that case, use first-success-step as a secondary metric.

In [ ]:
test_summary = summary[summary["group"] == "test"].copy()

# Sort by highest success rate, then lowest first success step
best = (
    test_summary.sort_values(
        ["config_name", "mean_success_rate", "mean_first_success_step"],
        ascending=[True, False, True],
    )
    .groupby("config_name")
    .head(1)
    [[
        "config_name",
        "checkpoint_step",
        "mean_success_rate",
        "std_success_rate",
        "mean_first_success_step",
        "mean_return",
        "n_runs",
    ]]
    .reset_index(drop=True)
)

best

## Save figures

In [ ]:
fig_dir = EVAL_DIR / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

def save_success_plot(group_name):
    fig, ax = plt.subplots(figsize=(10, 6))
    data = summary[summary["group"] == group_name]
    for config in sorted(data["config_name"].unique()):
        d = data[data["config_name"] == config].sort_values("checkpoint_step")
        ax.plot(d["checkpoint_step"], d["mean_success_rate"], marker="o", label=config)
        ax.errorbar(d["checkpoint_step"], d["mean_success_rate"], yerr=d["std_success_rate"].fillna(0), fmt="none", capsize=3)
    ax.set_title(f"Button-press-v3 success rate over checkpoints ({group_name})")
    ax.set_xlabel("Checkpoint timesteps")
    ax.set_ylabel("Mean success rate across splits")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend()
    out = fig_dir / f"button_checkpoint_success_rate_{group_name}.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return out

def save_first_success_plot(group_name):
    fig, ax = plt.subplots(figsize=(10, 6))
    data = summary[summary["group"] == group_name]
    for config in sorted(data["config_name"].unique()):
        d = data[data["config_name"] == config].sort_values("checkpoint_step")
        ax.plot(d["checkpoint_step"], d["mean_first_success_step"], marker="o", label=config)
        ax.errorbar(d["checkpoint_step"], d["mean_first_success_step"], yerr=d["std_first_success_step"].fillna(0), fmt="none", capsize=3)
    ax.set_title(f"Button-press-v3 first success step over checkpoints ({group_name})")
    ax.set_xlabel("Checkpoint timesteps")
    ax.set_ylabel("Mean first success step across splits")
    ax.grid(True, alpha=0.3)
    ax.legend()
    out = fig_dir / f"button_checkpoint_first_success_step_{group_name}.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return out

saved = []
for g in sorted(summary["group"].unique()):
    saved.append(save_success_plot(g))
    saved.append(save_first_success_plot(g))

for p in saved:
    print("Saved:", p)